# 从 GPT-2 到现代模型

> 我们已经知道，Part 4 搭的 MiniGPT 用的是经典 Transformer 配方：Post-Norm、ReLU 激活的 FFN、LayerNorm 归一化。这个配方来自 2017 年的原始 Transformer 论文，GPT-2 沿用了它。
>
> 这一节，我们把 MiniGPT 的 Transformer Block 逐步升级成现代 LLM 使用的版本，理解每个改进「为什么」。

2017 年的 Transformer 是为机器翻译设计的——编码器读一句英文，解码器生成一句法文。后来大家发现，把解码器单独拿出来做语言模型，效果出奇地好。GPT-2 就是这么来的：它只保留了 Transformer 的解码器部分，搭配 Post-Norm、ReLU FFN、LayerNorm 这些原始零件。

GPT-2 的表现不错，1.5B 的规模在当时已经是巨型模型。但当规模继续涨到 GPT-3 的 175B 时，这套配方开始出问题。问题不是某一个零件坏了，而是这些零件的设计在浅层网络里无伤大雅，到了深层网络里被放大了。

以 Post-Norm 为例。Post-Norm 把 LayerNorm 放在残差连接之后——做完 Attention，加上残差，再过 Norm。4 层的 MiniGPT 里，梯度穿过 4 个 Norm 传回第一层，衰减不算严重。但在 32 层的模型里，梯度要穿过 32 个 Norm。每穿过一层，Norm 都会调整梯度的尺度，层数越多，信号衰减越严重。深层参数收不到有效的梯度更新，等于白训了。

激活函数也有类似的问题。ReLU 的规则很简单：正数不变，负数变成零。如果某个神经元在一次更新后输出了负数，ReLU 会让它变成零。更关键的是，ReLU 在负数区的梯度也是零——这个神经元从此收不到任何梯度，永久「死亡」了。浅层网络里死几个神经元问题不大，但在深层网络里，累积的死亡神经元会让模型的有效容量持续下降。

这两个例子指向同一件事：原始 Transformer 零件是为浅层网络设计的。把它们搬到深层网络里，一些在小规模下不显眼的弱点会被深度放大。解决的思路不是推翻 Transformer 架构，而是找到这些不适配的零件，换成更适合深层网络的版本。

LLaMA（2023）系统地做了这件事。它把归一化从 LayerNorm 换成 RMSNorm（省掉去均值步骤），把激活函数从 ReLU 换成 SwiGLU（加门控让信息选择性通过），把位置编码从正弦编码换成 RoPE（用旋转矩阵编码相对位置），同时把 Post-Norm 改成 Pre-Norm（让残差路径绕过归一化）。每个改动只改一小处，但合在一起，32 层甚至更深的网络都能稳定训练。

这之后，模型被部署到生产环境，新的瓶颈又出现了。推理时 KV Cache 占的显存太大——一个 7B 模型处理 128K 上下文时，KV Cache 可能比模型参数本身还大。GQA 让多个 Q 头共享一组 KV 头，把 KV Cache 缩小了数倍。DeepSeek-V2 的 MLA 更进一步，把整个 KV 信息流做低秩压缩——不再按完整的 d_head 维度存 K 和 V，而是存一个极小的中间向量，推理时再解压。QK-Norm 则是在更大规模训练时暴露的问题：Q 和 K 的模长如果不加约束，Attention 的 softmax 会退化成近似 one-hot，丧失聚合多个位置信息的能力。在 Attention 前对 Q 和 K 各做一次 RMSNorm，问题就解决了。

下面从 Part 4 的原始 Block 出发，逐一动手换掉这些零件。


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 0. 先回顾：Part 4 的 Block 长什么样？在这里动手改

先把你已经熟悉的 Part 4 Block 贴出来，我们就在它上面一个一个改。

In [2]:
# === 这就是 Part 4 的版本，我们接下来的「改造对象」 ===

class FeedForward_Old(nn.Module):
    """原始 FFN：两层 Linear，中间 ReLU"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock_Old(nn.Module):
    """Post-Norm + LayerNorm + ReLU FFN（Part 4 版本）"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = FeedForward_Old(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)  # ← 普通 LayerNorm
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        # Post-Norm: 先算子层，再 +残差，最后 Norm
        x = self.norm1(x + self.attention(x, x, x, need_weights=False)[0])
        x = self.norm2(x + self.ffn(x))
        return x

print("✅ 这是 Part 4 的版本。接下来我们逐个升级。")
print("升级路线: LayerNorm→RMSNorm → ReLU→SwiGLU → Post-Norm→Pre-Norm")

✅ 这是 Part 4 的版本。接下来我们逐个升级。
升级路线: LayerNorm→RMSNorm → ReLU→SwiGLU → Post-Norm→Pre-Norm


---

### 1. 改进一：LayerNorm → RMSNorm

#### 1.1 LayerNorm 到底在算什么？— 用 4 个数字手算一遍

假设一个 token 的向量是 `[1, 3, 5, 7]`（4 维，和我们 Part 4 的 d_model=4 一样）。

**LayerNorm 做的事**：把这个向量的均值变成 0，标准差变成 1。

就像给一群学生调分——不管原始分数多高多低，调完之后平均分 0 分，分数分散程度统一。

具体步骤：
```
1. 算均值  μ = (x₁ + x₂ + x₃ + x₄) / 4
2. 算方差  σ² = ((x₁-μ)² + (x₂-μ)² + (x₃-μ)² + (x₄-μ)²) / 4
3. 算标准差 σ = √σ²
4. 归一化 x' = (x - μ) / σ
5. 缩放   y = γ × x' + β
```

γ 和 β 是可学习的参数（让模型自己决定调完之后「放多大」「往哪偏」）。

下面用 `[1, 3, 5, 7]` 手算一遍：

In [3]:
# === LayerNorm 手工计算 ===
print("=== LayerNorm 手算：输入 x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Step 1: 均值
mu = x.mean()
print(f"Step 1 — 均值 μ = (1+3+5+7)/4 = {mu:.1f}")

# Step 2: 方差（这里除以 N，不是 N-1）
var = torch.mean((x - mu) ** 2)
print(f"Step 2 — 方差 σ² = ((1-4)²+(3-4)²+(5-4)²+(7-4)²)/4")
print(f"        = (9 + 1 + 1 + 9)/4 = {var:.1f}")

# Step 3: 标准差
sigma = torch.sqrt(var)
print(f"Step 3 — 标准差 σ = √{var:.1f} = {sigma:.4f}")

# Step 4: 归一化
x_norm = (x - mu) / sigma
print(f"Step 4 — 归一化: (x - 4)/{sigma:.4f}")
for i, (xi, xni) in enumerate(zip(x.tolist(), x_norm.tolist())):
    print(f"         x[{i}] = ({xi:.1f} - 4) / {sigma:.4f} = {xni: .4f}")

print(f"         归一化后: {[f'{v:.4f}' for v in x_norm.tolist()]}")
print(f"         均值: {x_norm.mean():.4f} (=0), 标准差: {x_norm.std(unbiased=False):.4f} (=1)")

# Step 5: 缩放（假设 γ=[1,1,1,1], β=[0,0,0,0]）
# 初始时 γ 全 1，β 全 0，所以输出就是 x_norm
print(f"Step 5 — 缩放: γ=1, β=0 时输出 = 归一化结果")
print(f"         γ 和 β 是随着训练学出来的，让模型自己决定怎么调")

=== LayerNorm 手算：输入 x = [1, 3, 5, 7] ===

Step 1 — 均值 μ = (1+3+5+7)/4 = 4.0
Step 2 — 方差 σ² = ((1-4)²+(3-4)²+(5-4)²+(7-4)²)/4
        = (9 + 1 + 1 + 9)/4 = 5.0
Step 3 — 标准差 σ = √5.0 = 2.2361
Step 4 — 归一化: (x - 4)/2.2361
         x[0] = (1.0 - 4) / 2.2361 = -1.3416
         x[1] = (3.0 - 4) / 2.2361 = -0.4472
         x[2] = (5.0 - 4) / 2.2361 =  0.4472
         x[3] = (7.0 - 4) / 2.2361 =  1.3416
         归一化后: ['-1.3416', '-0.4472', '0.4472', '1.3416']
         均值: 0.0000 (=0), 标准差: 1.0000 (=1)
Step 5 — 缩放: γ=1, β=0 时输出 = 归一化结果
         γ 和 β 是随着训练学出来的，让模型自己决定怎么调


#### 1.2 LayerNorm 的问题：多算了一个不需要的东西

注意 LayerNorm 算了两个统计量：
- **μ（均值）**：把数据中心移到 0
- **σ（标准差）**：把数据分散程度统一

有人做了实验：**如果把「去均值」这一步去掉，只做缩放，效果会变差吗？**

答案：**不会。** 因为后面的线性变换（W_Q, W_K 等）本身就可以学会处理有均值的数据。

那还留着均值计算干嘛？去掉它，每次就能省一半的计算。

#### 1.3 RMSNorm：只做缩放

RMSNorm 的核心公式：

```
RMS(x) = √( (x₁² + x₂² + x₃² + x₄²) / 4 )

RMSNorm(x) = x / RMS(x) × γ
```

**对比**：
```
LayerNorm: (x - μ) / σ  × γ + β    ← μ 和 σ 两个统计量
RMSNorm:    x / RMS(x) × γ          ← 只有 RMS 一个统计量
```

省掉的是：不用算均值，不用算 (x-μ) 的偏差平方（只需要算 x²）。

用同一个 `[1, 3, 5, 7]` 手算 RMSNorm：

In [4]:
# === RMSNorm 手工计算 ===
print("=== RMSNorm 手算：输入 x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# 唯一的一步：算 RMS
mean_sq = torch.mean(x ** 2)
rms = torch.sqrt(mean_sq)

print(f"Step 1 — 平方均值 = (1²+3²+5²+7²)/4")
print(f"          = (1+9+25+49)/4")
print(f"          = {84/4:.1f}")
print(f"    RMS = √{mean_sq:.1f} = {rms:.4f}")
print()

# RMSNorm 输出
x_rmsnorm = x / rms
print(f"Step 2 — RMSNorm = x / {rms:.4f}")
for i, (xi, xri) in enumerate(zip(x.tolist(), x_rmsnorm.tolist())):
    print(f"         x[{i}] = {xi:.1f} / {rms:.4f} = {xri:.4f}")

print(f"\nRMSNorm 结果: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}")

# 验证：RMSNorm 后的 RMS 应该是 1
rms_after = torch.sqrt(torch.mean(x_rmsnorm ** 2))
print(f"\n验证 — 归一化后的 RMS = {rms_after:.4f} (应该 = 1) ✅")
print(f"        注意：RMSNorm 后的均值 ≠ 0（均值 = {x_rmsnorm.mean():.4f}）")
print(f"        这就是和 LayerNorm 的区别——不去均值！")
print()

# 对比
ln = nn.LayerNorm(4, elementwise_affine=False)  # 关掉 γ,β 方便对比
x_ln = ln(x)
print(f"LayerNorm 结果: {[f'{v:.4f}' for v in x_ln.tolist()]}  均值={x_ln.mean():.4f}  std={x_ln.std(unbiased=False):.4f}")
print(f"RMSNorm  结果: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}  均值={x_rmsnorm.mean():.4f}  RMS={rms_after:.4f}")
print(f"\n→ 数值不同，但都起到了「标准化」的作用")
print(f"→ LayerNorm 强制均值=0，RMSNorm 不强制（但效果一样好）")

=== RMSNorm 手算：输入 x = [1, 3, 5, 7] ===

Step 1 — 平方均值 = (1²+3²+5²+7²)/4
          = (1+9+25+49)/4
          = 21.0
    RMS = √21.0 = 4.5826

Step 2 — RMSNorm = x / 4.5826
         x[0] = 1.0 / 4.5826 = 0.2182
         x[1] = 3.0 / 4.5826 = 0.6547
         x[2] = 5.0 / 4.5826 = 1.0911
         x[3] = 7.0 / 4.5826 = 1.5275

RMSNorm 结果: ['0.2182', '0.6547', '1.0911', '1.5275']

验证 — 归一化后的 RMS = 1.0000 (应该 = 1) ✅
        注意：RMSNorm 后的均值 ≠ 0（均值 = 0.8729）
        这就是和 LayerNorm 的区别——不去均值！

LayerNorm 结果: ['-1.3416', '-0.4472', '0.4472', '1.3416']  均值=0.0000  std=1.0000
RMSNorm  结果: ['0.2182', '0.6547', '1.0911', '1.5275']  均值=0.8729  RMS=1.0000

→ 数值不同，但都起到了「标准化」的作用
→ LayerNorm 强制均值=0，RMSNorm 不强制（但效果一样好）


In [5]:
# === RMSNorm 完整实现 ===

class RMSNorm(nn.Module):
    """
    RMSNorm: 只做缩放，不做中心化
    
    公式: y = x / RMS(x) × γ
    RMS(x) = √(mean(x²) + ε)
    
    - 没有 β（bias），因为不需要补偿去均值造成的偏移
    - ε 是为了防止除以 0（比如输入全 0）
    """
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # 只有 γ
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        # 沿最后一维算 RMS
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma


# 测试
d_model = 8
rmsn = RMSNorm(d_model)
ln = nn.LayerNorm(d_model)

batch = torch.randn(2, 4, d_model) * 3  # 模拟 2 个 batch, 4 个 token, 8 维
out_rms = rmsn(batch)
out_ln = ln(batch)

print("=== RMSNorm 实现验证 ===")
print(f"输入形状: {batch.shape}")
print(f"RMSNorm 输出形状: {out_rms.shape}")
print(f"输出 RMS（应该 ≈ 1）: {torch.sqrt(torch.mean(out_rms**2, dim=-1))[0].tolist()}")
print()

# 参数量对比
ln_params = sum(p.numel() for p in ln.parameters())
rmsn_params = sum(p.numel() for p in rmsn.parameters())
print(f"LayerNorm 参数: γ({d_model}) + β({d_model}) = {ln_params}")
print(f"RMSNorm  参数: γ({d_model}) = {rmsn_params}")
print(f"→ 参数减半，但这不是重点——重点是计算时少算了均值")

=== RMSNorm 实现验证 ===
输入形状: torch.Size([2, 4, 8])
RMSNorm 输出形状: torch.Size([2, 4, 8])
输出 RMS（应该 ≈ 1）: [0.9999999403953552, 0.9999998807907104, 1.0, 0.9999999403953552]

LayerNorm 参数: γ(8) + β(8) = 16
RMSNorm  参数: γ(8) = 8
→ 参数减半，但这不是重点——重点是计算时少算了均值


---

### 2. 改进二：ReLU → SwiGLU

#### 2.1 先看原始 FFN 里 ReLU 做了什么

Part 4 的 FFN 公式：

```
FFN(x) = W₂ · ReLU(W₁ · x)

ReLU 做的事：负数 → 0，正数 → 原样通过
```

用一个具体的向量手算看看：

In [6]:
# === ReLU 手算 ===
print("=== ReLU 对每个数字做了什么 ===")
print()

# 模拟 W₁·x 的输出（扩展到 8 维）
hidden = torch.tensor([0.5, -2.0, 3.0, -0.1, 0.0, -5.0, 1.5, -0.01])

print(f"输入 (W₁·x 的结果): {[f'{v:.2f}' for v in hidden.tolist()]}")
print()

relu_out = F.relu(hidden)
print("ReLU 做了什么：")
for i, (h, r) in enumerate(zip(hidden.tolist(), relu_out.tolist())):
    action = "保留" if h >= 0 else "→ 0 (杀死)"
    print(f"  位置 {i}: {h:6.2f} → {r:6.2f}  {action}")

print(f"\n问题：8 个位置中有 {sum(1 for h in hidden if h < 0)} 个被直接杀死了")
print(f"      其中 -0.1 和 -0.01 只是「稍微负了一点」，也被杀了")
print(f"      → ReLU 太粗暴：负的全部不要")

=== ReLU 对每个数字做了什么 ===

输入 (W₁·x 的结果): ['0.50', '-2.00', '3.00', '-0.10', '0.00', '-5.00', '1.50', '-0.01']

ReLU 做了什么：
  位置 0:   0.50 →   0.50  保留
  位置 1:  -2.00 →   0.00  → 0 (杀死)
  位置 2:   3.00 →   3.00  保留
  位置 3:  -0.10 →   0.00  → 0 (杀死)
  位置 4:   0.00 →   0.00  保留
  位置 5:  -5.00 →   0.00  → 0 (杀死)
  位置 6:   1.50 →   1.50  保留
  位置 7:  -0.01 →   0.00  → 0 (杀死)

问题：8 个位置中有 4 个被直接杀死了
      其中 -0.1 和 -0.01 只是「稍微负了一点」，也被杀了
      → ReLU 太粗暴：负的全部不要


#### 2.2 门控的直觉：编辑 vs 总编

想象你在写一篇文章，然后交给编辑部审稿：

- **ReLU 方式（一个编辑）**：编辑看完每一段，要么全文通过（正），要么全文删除（负）。一段话如果有 90% 好 + 10% 有瑕疵 → 整个删了，可惜。

- **门控方式（两个编辑）**：
  - 编辑 A：负责理解内容（信息通道）
  - 编辑 B：负责打分（门控通道）——这段有多重要？给 0~1 之间的分数
  - 最终输出 = A 的理解 × B 的分数

效果：如果一段话「稍微有点问题」，B 给 0.3 分，它还是能通过一部分——而不是完全消失。

公式就长这样：

```
SwiGLU(x) = (W_up · x)  ⊙  Swish(W_gate · x)
              ↑ 信息通道       ↑ 门控通道
              
Swish(a) = a × sigmoid(a) = a / (1 + e^(-a))
```

**Swish 是什么？** 它是一个平滑的「软开关」——不像 ReLU 那样一刀切。

In [7]:
# === ReLU vs Swish 手算对比 ===
print("=== 激活函数对比：ReLU vs Swish ===")
print()

x_vals = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

print(f"{'x':>8s}  {'ReLU(x)':>10s}  {'Swish(x)':>10s}  {'说明'}")
print("-" * 52)

for x in x_vals:
    relu = max(0, x)
    swish = x / (1 + math.exp(-x))  # x * sigmoid(x)
    note = ""
    if x < 0:
        note = f"ReLU 杀死, Swish 保留 {swish:.4f}"
    elif x == 0:
        note = "ReLU 卡在 0, Swish 平滑"
    else:
        note = "两者都通过"
    print(f"{x:>8.1f}  {relu:>10.1f}  {swish:>10.4f}  {note}")

print()
print("关键观察：")
print("  1. 正数区: Swish ≈ ReLU（略小一点），行为相似")
print("  2. 负数区: ReLU = 0（彻底杀死），Swish 保留一个小负值")
print("  3. x=0 处: ReLU 不可导（数学上），Swish 光滑可导")
print()
print("为什么「小负值」重要？")
print("  梯度 = ∂loss/∂x，而 ∂Swish/∂x 在负数区不为 0")
print("  → 即使这个神经元当前「不太激活」，梯度仍然可以流过")
print("  → ReLU 的神经元可能「永久死亡」（一旦进入负数区，梯度=0，再也回不来）")

=== 激活函数对比：ReLU vs Swish ===

       x     ReLU(x)    Swish(x)  说明
----------------------------------------------------
    -3.0         0.0     -0.1423  ReLU 杀死, Swish 保留 -0.1423
    -2.0         0.0     -0.2384  ReLU 杀死, Swish 保留 -0.2384
    -1.0         0.0     -0.2689  ReLU 杀死, Swish 保留 -0.2689
    -0.5         0.0     -0.1888  ReLU 杀死, Swish 保留 -0.1888
     0.0         0.0      0.0000  ReLU 卡在 0, Swish 平滑
     0.5         0.5      0.3112  两者都通过
     1.0         1.0      0.7311  两者都通过
     2.0         2.0      1.7616  两者都通过
     3.0         3.0      2.8577  两者都通过

关键观察：
  1. 正数区: Swish ≈ ReLU（略小一点），行为相似
  2. 负数区: ReLU = 0（彻底杀死），Swish 保留一个小负值
  3. x=0 处: ReLU 不可导（数学上），Swish 光滑可导

为什么「小负值」重要？
  梯度 = ∂loss/∂x，而 ∂Swish/∂x 在负数区不为 0
  → 即使这个神经元当前「不太激活」，梯度仍然可以流过
  → ReLU 的神经元可能「永久死亡」（一旦进入负数区，梯度=0，再也回不来）


In [8]:
# === 手动计算 SwiGLU 的一步 ===
print("=== SwiGLU 手算 ===")
print()

# 用极小例子：d_model=4, 模拟一条信息
d_model = 4
x = torch.tensor([[1.0, -0.5, 2.0, 0.3]])  # 一个 token

# 模拟三个权重矩阵的简单版本（手工构造便于观察）
# 实际中这些是 nn.Linear 的权重
torch.manual_seed(1)
W_up = nn.Linear(d_model, d_model, bias=False)
W_gate = nn.Linear(d_model, d_model, bias=False)
W_down = nn.Linear(d_model, d_model, bias=False)

# 信息通道
up = W_up(x)
# 门控通道：先线性变换，再过 Swish
gate = F.silu(W_gate(x))  # silu = Swish
# 门控结果 = 信息 × 门
gated = up * gate
# 输出投影
out = W_down(gated)

print(f"输入 x: {x.tolist()}")
print()
print(f"信息通道 (W_up·x):     {[f'{v:.3f}' for v in up[0].tolist()]}")
print(f"门控通道 Swish(W_gate·x): {[f'{v:.3f}' for v in gate[0].tolist()]}")
print(f"门控后 (up × gate):     {[f'{v:.3f}' for v in gated[0].tolist()]}")
print(f"输出 (W_down·gated):    {[f'{v:.3f}' for v in out[0].tolist()]}")
print()
print("门控的效果: 如果 gate 某个位置 ≈ 0 → 信息通道对应位置被「关门」→ 信息传不过去")
print("           如果 gate 某个位置 ≈ 1 → 信息通道对应位置「开门」→ 信息完全通过")
print("           如果 gate 某个位置 = 0.3 → 信息只通过 30%")

=== SwiGLU 手算 ===

输入 x: [[1.0, -0.5, 2.0, 0.30000001192092896]]

信息通道 (W_up·x):     ['0.245', '-0.750', '0.385', '0.194']
门控通道 Swish(W_gate·x): ['0.726', '-0.116', '0.315', '-0.099']
门控后 (up × gate):     ['0.177', '0.087', '0.121', '-0.019']
输出 (W_down·gated):    ['-0.034', '0.101', '0.114', '-0.059']

门控的效果: 如果 gate 某个位置 ≈ 0 → 信息通道对应位置被「关门」→ 信息传不过去
           如果 gate 某个位置 ≈ 1 → 信息通道对应位置「开门」→ 信息完全通过
           如果 gate 某个位置 = 0.3 → 信息只通过 30%


In [9]:
# === 完整 SwiGLU FFN 实现 ===

class FeedForward_SwiGLU(nn.Module):
    """
    SwiGLU FFN（LLaMA / DeepSeek 同款）
    
    公式: FFN(x) = W_down · ( Swish(W_gate·x) ⊙ (W_up·x) )
    
    三个权重矩阵：
    - W_gate: 门控投影 (d_model → d_ff)
    - W_up:   信息投影 (d_model → d_ff)
    - W_down: 输出投影 (d_ff → d_model)
    
    注意：原始 FFN 有 2 个矩阵，SwiGLU 有 3 个。为了保持参数量一致，
    d_ff 要调小：原始 d_ff=4d，SwiGLU d_ff ≈ 8/3 d
    """
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            # 3 * d_model * d_ff ≈ 2 * d_model * 4d_model
            # → d_ff ≈ 8/3 * d_model
            d_ff = int(8 / 3 * d_model)
            d_ff = ((d_ff + 255) // 256) * 256  # 取整到 256 的倍数
            d_ff = max(d_ff, d_model)  # 至少和 d_model 一样大
        
        self.W_gate = nn.Linear(d_model, d_ff, bias=False)
        self.W_up = nn.Linear(d_model, d_ff, bias=False)
        self.W_down = nn.Linear(d_ff, d_model, bias=False)
    
    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))


# 参数量对比
d_model = 512
ffn_old = FeedForward_Old(d_model)
ffn_new = FeedForward_SwiGLU(d_model)

old_p = sum(p.numel() for p in ffn_old.parameters())
new_p = sum(p.numel() for p in ffn_new.parameters())

print("=== SwiGLU vs ReLU FFN 参数量对比 ===")
print(f"d_model = {d_model}")
print()
print(f"ReLU FFN:  W1({d_model}×{4*d_model}) + W2({4*d_model}×{d_model})")
print(f"           = {d_model*4*d_model:,} + {4*d_model*d_model:,}")
print(f"           = {old_p:,} 个参数")
print()
d_ff_s = ((int(8/3*d_model) + 255) // 256) * 256
print(f"SwiGLU FFN: W_gate({d_model}×{d_ff_s}) + W_up({d_model}×{d_ff_s}) + W_down({d_ff_s}×{d_model})")
print(f"           = {d_model*d_ff_s:,} + {d_model*d_ff_s:,} + {d_ff_s*d_model:,}")
print(f"           = {new_p:,} 个参数")
print()
print(f"参数量比: {new_p/old_p:.2f}x (≈1 即可)")
print(f"→ 参数差不多，但 SwiGLU 效果好很多")

=== SwiGLU vs ReLU FFN 参数量对比 ===
d_model = 512

ReLU FFN:  W1(512×2048) + W2(2048×512)
           = 1,048,576 + 1,048,576
           = 2,099,712 个参数

SwiGLU FFN: W_gate(512×1536) + W_up(512×1536) + W_down(1536×512)
           = 786,432 + 786,432 + 786,432
           = 2,359,296 个参数

参数量比: 1.12x (≈1 即可)
→ 参数差不多，但 SwiGLU 效果好很多


---

### 3. 改进三：Post-Norm → Pre-Norm

#### 3.1 先搞清 Post-Norm 的「Post」是什么意思

Part 4 的 Block 里，写法是这样：

```python
x = x + attention(x)   # 残差：把输入和 Attention 输出加起来
x = norm(x)            # LayerNorm：标准化
```

这里 LayerNorm 在 Attention **之后**，所以叫 **Post-Norm**。

完整的计算图（Post-Norm）：

```
  x ──→ Attention ──→ + ──→ LayerNorm ──→ 输出
  │                    ↑
  └────────────────────┘  (残差连接)
```

注意：残差连接加到 Attention 的输出上，然后两者一起被 LayerNorm。
**这意味着残差路径经过了 LayerNorm！**

#### 3.2 这有什么问题？

回忆 Part 4，我们说过残差连接是「梯度高速公路」——梯度可以通过残差路径直接回传，不被 Attention 和 FFN 衰减。

但现在，在 Post-Norm 里，**这条高速公路上多了一个收费站——LayerNorm**。

LayerNorm 会缩放梯度，这意味着经过 LayerNorm 后，梯度的大小可能被改变。

在 4 层网络里这不是问题。但在 40 层、80 层网络里，每一层都经过一次 LayerNorm → 梯度的缩放效应会累积 → 可能爆炸或消失。

#### 3.3 Pre-Norm：收费站搬到辅路上

Pre-Norm 的做法：把 LayerNorm 搬到 Attention **之前**：

```python
x = x + attention(norm(x))   # 先 Norm，再做 Attention
```

计算图（Pre-Norm）：

```
  x ──→ LayerNorm ──→ Attention ──→ + ──→ 输出
  │                                   ↑
  └───────────────────────────────────┘  (残差连接，不经过 Norm！)
```

**关键变化**：残差连接绕过了 LayerNorm！梯度在高速公路上没有任何收费站。

用两个极小的网络对比：

In [10]:
# === Post-Norm vs Pre-Norm：用手算看梯度流动 ===
print("=== Post-Norm vs Pre-Norm：梯度流动对比 ===")
print()

# 搭两个简化的深层网络（用 FFN 代替 Attention，专注看 Norm 位置的影响）
d_model = 4
num_layers = 8  # 先用 8 层看效果

class Deep_PostNorm(nn.Module):
    """Post-Norm: x = Norm(x + FFN(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = norm(x + F.relu(layer(x)))
        return x

class Deep_PreNorm(nn.Module):
    """Pre-Norm: x = x + FFN(Norm(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = x + F.relu(layer(norm(x)))
        return x

# 相同初始化
torch.manual_seed(42)
model_post = Deep_PostNorm(d_model, num_layers)
torch.manual_seed(42)
model_pre = Deep_PreNorm(d_model, num_layers)

# 随机输入
x = torch.randn(2, d_model)
target = torch.randn(2, d_model)

# Forward + Backward
loss_post = F.mse_loss(model_post(x), target)
loss_post.backward()

loss_pre = F.mse_loss(model_pre(x), target)
loss_pre.backward()

# 看每一层的梯度范数
print(f"网络深度: {num_layers} 层")
print()
print(f"{'层':>6s}  {'Post-Norm 梯度':>16s}  {'Pre-Norm 梯度':>16s}")
print("-" * 42)

for i in range(num_layers):
    grad_post = model_post.layers[i].weight.grad.norm().item()
    grad_pre = model_pre.layers[i].weight.grad.norm().item()
    print(f"Layer {i+1}:  {grad_post:>16.6f}  {grad_pre:>16.6f}")

# 最重要：看第 1 层的梯度（最底层）
grad_post_first = model_post.layers[0].weight.grad.norm().item()
grad_pre_first = model_pre.layers[0].weight.grad.norm().item()

print()
print(f"最关键——最底层（Layer 1）梯度对比:")
print(f"  Post-Norm Layer 1 梯度: {grad_post_first:.6f}")
print(f"  Pre-Norm  Layer 1 梯度: {grad_pre_first:.6f}")
print(f"  Pre-Norm / Post-Norm = {grad_pre_first/grad_post_first:.2f}x")
print()
print("→ Pre-Norm 底层梯度更大，因为残差路径绕过了所有 LayerNorm")
print("→ 在 80 层网络里，这个差别会大到 Post-Norm 根本训不动")

=== Post-Norm vs Pre-Norm：梯度流动对比 ===

网络深度: 8 层

     层      Post-Norm 梯度       Pre-Norm 梯度
------------------------------------------
Layer 1:          0.411592          6.629198
Layer 2:          0.319549          5.358179
Layer 3:          0.309842          6.145226
Layer 4:          0.317241          5.256918
Layer 5:          0.202899          3.324109
Layer 6:          0.217564          2.635670
Layer 7:          0.114408          3.379233
Layer 8:          0.532629          4.577710

最关键——最底层（Layer 1）梯度对比:
  Post-Norm Layer 1 梯度: 0.411592
  Pre-Norm  Layer 1 梯度: 6.629198
  Pre-Norm / Post-Norm = 16.11x

→ Pre-Norm 底层梯度更大，因为残差路径绕过了所有 LayerNorm
→ 在 80 层网络里，这个差别会大到 Post-Norm 根本训不动


#### 3.4 用做饭类比总结

- **Post-Norm**：先炒菜，再尝味道 → 如果炒糊了，调整也没用（梯度已经被 Norm 截断了）
- **Pre-Norm**：先尝调料确认味道，再炒 → 炒的过程中信号始终稳定（残差路径畅通）

所以在深层网络中，Pre-Norm 的训练稳定得多——它可以用更大的学习率，更少的 warmup。

---

### 4. 拼起来！现代 LLaMA-style Block

三个改进全部到位，把它们一次组装：

```
LLaMA Block = Pre-Norm (RMSNorm) + Attention + Pre-Norm (RMSNorm) + SwiGLU FFN

  x
  │
  ├─→ RMSNorm → Attention ─→ +  ← 残差（不经过 Norm！）
  │                           │
  ├───────────────────────────┘
  │
  ├─→ RMSNorm → SwiGLU FFN ─→ +  ← 残差（不经过 Norm！）
  │                            │
  └────────────────────────────┘
  ↓
 输出
```

In [11]:
class LLaMABlock(nn.Module):
    """
    现代 LLM 的 Transformer Block
    
    三大升级 vs Part 4:
    1. LayerNorm → RMSNorm
    2. Post-Norm → Pre-Norm
    3. ReLU FFN → SwiGLU FFN
    """
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        # Pre-Norm: RMSNorm 在 Attention 前面
        self.norm_attn = RMSNorm(d_model)
        # Multi-Head Self-Attention（和 Part 4 一样）
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        # Pre-Norm: RMSNorm 在 FFN 前面
        self.norm_ffn = RMSNorm(d_model)
        # SwiGLU FFN
        self.ffn = FeedForward_SwiGLU(d_model, d_ff)
    
    def forward(self, x, mask=None):
        # 注意顺序：先 Norm，再子层，再残差
        h = self.norm_attn(x)
        attn_out, _ = self.attention(h, h, h, attn_mask=mask, need_weights=False)
        x = x + attn_out  # 残差：绕过 Norm！
        
        h = self.norm_ffn(x)
        x = x + self.ffn(h)  # 残差：绕过 Norm！
        
        return x

# 测试
d_model, num_heads = 8, 2
block_new = LLaMABlock(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)
causal_mask = torch.triu(torch.ones(5, 5) * float('-inf'), diagonal=1)

out_new = block_new(test_x, causal_mask)
print("=== LLaMA Block 测试 ===")
print(f"输入形状: {test_x.shape}")
print(f"输出形状: {out_new.shape}  ← 不变！")
print(f"但里面的组件全部升级了:")
print(f"  ✓ RMSNorm 替代 LayerNorm")
print(f"  ✓ Pre-Norm 替代 Post-Norm")
print(f"  ✓ SwiGLU 替代 ReLU FFN")

=== LLaMA Block 测试 ===
输入形状: torch.Size([1, 5, 8])
输出形状: torch.Size([1, 5, 8])  ← 不变！
但里面的组件全部升级了:
  ✓ RMSNorm 替代 LayerNorm
  ✓ Pre-Norm 替代 Post-Norm
  ✓ SwiGLU 替代 ReLU FFN


### 5. 三代 Transformer Block 演进全景图

```
2017  原始 Transformer (Vaswani et al.)
      Post-Norm + LayerNorm + ReLU FFN
      └→ 能跑，但深层不好训（原始论文只用了 6 层）

2019  GPT-2 / GPT-3
      Pre-Norm + LayerNorm + GELU FFN
      └→ 改了 Norm 位置，能堆到 96 层了
          GELU 比 ReLU 好一点，但还是不够好

2023  LLaMA 2 / 3, DeepSeek, Qwen, Mistral...
      Pre-Norm + RMSNorm + SwiGLU FFN
      └→ 三个改进全用上，训练又快又稳效果又好
```

如果要给三个改进的重要性排序：
1. **Pre-Norm**（最重要——决定了能不能训深层）
2. **SwiGLU**（效果明显——门控机制让 FFN 更聪明）
3. **RMSNorm**（锦上添花——省了计算，但换了也不影响大局）

## 从 GPT-2 到现代模型小结

确认你已经搞懂了这些（按顺序检查）：

1. ✅ LayerNorm = (x - μ) / σ × γ + β，算两个统计量
2. ✅ RMSNorm = x / RMS(x) × γ，只算一个统计量——去掉了去均值
3. ✅ RMSNorm 省的计算：不用算 μ，不用算 (x-μ)²（直接用 x²）
4. ✅ ReLU 的问题：负数直接杀 → 梯度断掉，神经元可能死亡
5. ✅ SwiGLU = 信息通道 (W_up) × 门控通道 (Swish(W_gate))→ 选择性通过
6. ✅ Swish 平滑且负数区有非零梯度 → 不会「杀死」神经元
7. ✅ Post-Norm = 子层+残差→Norm，残差路径经过 Norm
8. ✅ Pre-Norm = Norm→子层→+残差，残差路径绕过 Norm
9. ✅ Pre-Norm 好在哪：梯度高速公路无收费站 → 深层也能训
10. ✅ LLaMA Block = Pre-Norm + RMSNorm + SwiGLU（现代 LLM 标准配方）
11. ✅ GQA = 多个 Q 头共享一组 KV 头，KV Cache 大幅缩小
12. ✅ MHA（32 KV 头）→ GQA（4 KV 头，省 87.5%）→ MQA（1 KV 头，省 97%）
13. ✅ GQA 实现关键：K/V 投影更小，expand 到和 Q 一样再做 attention
14. ✅ QK-Norm = 算 attention 前对 Q、K 各自 RMSNorm → 防止 softmax 退化
15. ✅ QK-Norm 只用两行代码，LLaMA 3 / Qwen 3 标配
16. ✅ MLA = KV 低秩压缩到 latent 空间，先压到极小的 c_KV，推理时再解压
17. ✅ MLA 的 KV Cache 比 MHA 降一个数量级，DeepSeek-V2/V3 和 Kimi K2 的核心设计

**一句话总结**：现代 Decoder-only 模型不是推翻 GPT-2，而是在同一条主干上升级关键零件——Pre-Norm 解决「能不能训」（稳定性）、SwiGLU 解决「训得好不好」（效果）、RMSNorm 解决「训得快不快」（效率）、GQA 解决「推理贵不贵」（KV Cache）、QK-Norm 解决「训得稳不稳」（退化）、MLA 解决「Cache 还能不能更小」（终极压缩）。

### 6. 改进四：MHA → GQA — Attention 的 KV 太贵了

前面三个改进解决了「稳定性」「效果」「效率」。还有一个工程问题没解决：

**Attention 每一步都要把之前所有 token 的 K 和 V 存下来（KV Cache），序列越长，显存占用越大。**

这个问题在推理时尤其严重——用户给了一个 32K 的上下文，模型要为每个注意力头都存一份 K 和 V。

#### 6.1 MHA、MQA、GQA 分别是什么

以 LLaMA 2 7B 为例：32 个注意力头，每个头的维度 128。

| 方案 | Q 的头数 | K 的头数 | V 的头数 | KV Cache 大小 | 谁在用 |
|:---|:---|:---|:---|:---|:---|
| **MHA** | 32 | 32 | 32 | 100% | GPT-2/3, BLOOM |
| **MQA** | 32 | **1** | **1** | **3%** | PaLM, StarCoder |
| **GQA** | 32 | **4** | **4** | **12.5%** | LLaMA 2/3, Mistral |

直觉：
- **MHA**（Multi-Head Attention）：每个 Q 头有自己的 K 和 V。像 32 个人各自看书，每人配一个助理。
- **MQA**（Multi-Query Attention）：32 个 Q 头共享 1 组 K 和 V。32 个人共用一个助理——省人，但可能忙不过来。
- **GQA**（Grouped-Query Attention）：32 个 Q 头分成 4 组，每组共享 1 组 K 和 V。4 个助理服务 32 个人——折中方案。

GQA 是目前最流行的方案，因为它在 MHA 和 MQA 之间找到了平衡点：KV Cache 大幅缩小，但效果几乎不掉。

In [ ]:
# === MHA vs GQA vs MQA：KV Cache 手算对比 ===
print("=== KV Cache 对比：MHA / GQA / MQA ===")
print()

# 假设：LLaMA 2 7B 的参数
num_q_heads = 32      # Q 的头数
d_head = 128          # 每个头的维度
seq_len = 4096        # 序列长度
dtype_bytes = 2       # bf16 占 2 字节

# 三种方案的 KV 头数
configs = [
    ("MHA", num_q_heads, num_q_heads),
    ("GQA-4", 4, num_q_heads),
    ("GQA-8", 8, num_q_heads),
    ("MQA", 1, num_q_heads),
]

print(f"模型配置: {num_q_heads} 个 Q 头, 每头 {d_head} 维")
print(f"序列长度: {seq_len}, 数据类型: bf16 (2 bytes)")
print()

mha_size = None

for name, kv_heads, _ in configs:
    # KV Cache 大小 = 2(K+V) × kv_heads × d_head × seq_len × dtype_bytes
    kv_size = 2 * kv_heads * d_head * seq_len * dtype_bytes
    kv_size_mb = kv_size / (1024 ** 2)
    
    if mha_size is None:
        mha_size = kv_size
    ratio = kv_size / mha_size * 100
    
    # 每组多少个 Q 头
    q_per_group = num_q_heads // kv_heads
    
    print(f"{name:<8s} KV头数={kv_heads:>2d}  "
          f"每组Q头={q_per_group:>2d}  "
          f"KV Cache={kv_size_mb:>6.1f} MB  "
          f"({ratio:>5.1f}%)")

print()
print("关键数字：")
print(f"  MHA:   {2 * num_q_heads * d_head * seq_len * dtype_bytes / (1024**2):.0f} MB（32 个 KV 头）")
print(f"  GQA-4: {2 * 4 * d_head * seq_len * dtype_bytes / (1024**2):.0f} MB（4 个 KV 头，省 87.5%）")
print(f"  MQA:   {2 * 1 * d_head * seq_len * dtype_bytes / (1024**2):.0f} MB（1 个 KV 头，省 97%）")
print()
print("LLaMA 2 7B 用 GQA-4，KV Cache 只有 MHA 的 1/8")
print("这意味着同样的显存可以处理 8 倍长的序列，或服务 8 倍多的并发请求")

#### 6.2 GQA 的实现：关键就一个 reshape

GQA 和 MHA 的区别只在一步：把 Q 按「组」切分，每组内的 Q 头共享同一组 K 和 V。

```
MHA:  Q [32, d] × K [32, d] → 32 组独立的 attention
GQA:  Q [32, d] × K [4, d]  → 32 个 Q 分成 4 组，每组 8 个 Q 共享 1 个 K

实现方式：
  1. K/V 只有 4 个头，通过 repeat 扩展到 32 个头
  2. 然后和 MHA 完全一样做 attention
```

所以 GQA 的代码改动极小——只需在 K、V 上多做一次 `expand`。

In [ ]:
# === GQA 实现演示 ===

class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention (GQA)
    
    参数:
        d_model: 模型维度
        num_q_heads: Q 的头数（如 32）
        num_kv_heads: K/V 的头数（如 4），必须能整除 num_q_heads
    """
    def __init__(self, d_model, num_q_heads, num_kv_heads):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0
        
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.d_head = d_model // num_q_heads
        self.q_per_group = num_q_heads // num_kv_heads  # 每组几个 Q 头
        
        # Q 有 num_q_heads 个头，K/V 只有 num_kv_heads 个头
        self.W_q = nn.Linear(d_model, num_q_heads * self.d_head, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(num_q_heads * self.d_head, d_model, bias=False)
    
    def forward(self, x, mask=None):
        B, S, D = x.shape
        
        # 投影
        q = self.W_q(x)  # [B, S, num_q * d_head]
        k = self.W_k(x)  # [B, S, num_kv * d_head]
        v = self.W_v(x)  # [B, S, num_kv * d_head]
        
        # reshape 成多头形式
        q = q.view(B, S, self.num_q_heads, self.d_head)
        k = k.view(B, S, self.num_kv_heads, self.d_head)
        v = v.view(B, S, self.num_kv_heads, self.d_head)
        
        # GQA 关键一步：把 K/V 扩展到和 Q 一样的头数
        # 每个 KV 头被 self.q_per_group 个 Q 头共享
        k = k[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)
        
        v = v[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)
        
        # 转成 [B, heads, S, d_head] 做 attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # 缩放点积 attention（和 Part 4 一样）
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores + mask
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        
        # 合并头，输出投影
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)


# 测试：对比 MHA 和 GQA
d_model = 256
num_q = 8    # 8 个 Q 头
num_kv = 2   # 2 个 KV 头 → 每组 4 个 Q 共享 1 个 KV

gqa = GroupedQueryAttention(d_model, num_q, num_kv)
mha = nn.MultiheadAttention(d_model, num_q, batch_first=True)

x = torch.randn(1, 6, d_model)
mask = torch.triu(torch.ones(6, 6) * float('-inf'), diagonal=1)

out_gqa = gqa(x, mask)
out_mha, _ = mha(x, x, x, attn_mask=mask, need_weights=False)

print("=== GQA 实现 ===")
print(f"输入: {x.shape}")
print(f"GQA 输出: {out_gqa.shape}")
print(f"MHA 输出: {out_mha.shape}")
print()

# 参数量对比
gqa_params = sum(p.numel() for p in gqa.parameters())
mha_params = sum(p.numel() for p in mha.parameters())
print(f"GQA 参数: {gqa_params:,}  (KV 投影更小)")
print(f"MHA 参数: {mha_params:,}")
print(f"GQA 参数是 MHA 的 {gqa_params/mha_params:.2%}")
print()
print(f"KV 投影对比:")
print(f"  MHA: W_k {d_model}×{d_model} = {d_model*d_model:,}")
print(f"  GQA: W_k {d_model}×{num_kv * (d_model//num_q)} = {d_model * num_kv * (d_model//num_q):,}")
print(f"  → GQA 的 KV 投影只有 MHA 的 {num_kv/num_q:.0%}")

### 7. 改进五：QK-Norm — 防止 Attention 退化

GQA 解决了 KV Cache 的大小，但 Attention 还有一个隐蔽的问题：训练到后期，Q（Query）和 K（Key）的向量模长可能差异极大。

举个例子：

```text
Q = [0.2, -0.1, 0.3, ...] → 模长 ≈ 1.5
K = [12.3, -8.7, 5.2, ...] → 模长 ≈ 25.0
```

这时候 `Q @ K^T` 的某一项可能变成 `1.5 × 25.0 = 37.5`，而其他项只有个位数。Softmax 遇到这种悬殊的输入会退化成近似 one-hot——模型只关注一个 token，丧失了 Attention 聚合多个位置信息的能力。LLaMA 3 和 Qwen 3 的技术报告里都记录了同一个现象：attention logit 的数值范围随训练步数持续增长，最终导致训练不稳定。

**修复方式**：在算 Attention 之前，先对 Q 和 K 各自做 RMSNorm。

```python
# 原来的 MHA:
Q = W_Q(x)  # Q 的模长任意
K = W_K(x)  # K 的模长任意
score = Q @ K^T / sqrt(d_head)

# QK-Norm:
Q = RMSNorm(W_Q(x))  # Q 的模长被拉到 ≈ 1
K = RMSNorm(W_K(x))  # K 的模长被拉到 ≈ 1
score = Q @ K^T / sqrt(d_head)
```

RMSNorm 之后，Q 和 K 的每个头的模长都在 1 附近，`Q @ K^T` 的值域被限制在 `[-1, 1]` 范围内，Softmax 不会退化成 one-hot。改动极小：只加两行 RMSNorm，不改变 Attention 的任何其他逻辑。


In [ ]:
# === QK-Norm 对比：不加 vs 加 ===
print("=== QK-Norm：模拟 Attention logit 退化问题 ===")
print()

# 模拟训练后期的 Q 和 K：模长差异大
torch.manual_seed(42)
d_head = 8
seq_len = 4

# 正常的 Q
Q_normal = torch.randn(1, 1, seq_len, d_head) * 1.0
# 训练后期 K 的模长变得很大
K_large = torch.randn(1, 1, seq_len, d_head) * 8.0

# 不用 QK-Norm：直接算 attention score
score_no_norm = (Q_normal @ K_large.transpose(-2, -1)) / math.sqrt(d_head)
attn_no_norm = F.softmax(score_no_norm, dim=-1)

# 用 QK-Norm：先各自 RMSNorm
Q_normed = Q_normal / torch.sqrt(torch.mean(Q_normal ** 2, dim=-1, keepdim=True) + 1e-6)
K_normed = K_large / torch.sqrt(torch.mean(K_large ** 2, dim=-1, keepdim=True) + 1e-6)
score_with_norm = (Q_normed @ K_normed.transpose(-2, -1)) / math.sqrt(d_head)
attn_with_norm = F.softmax(score_with_norm, dim=-1)

print("不加 QK-Norm:")
print(f"  Q 模长范围: {Q_normal.norm(dim=-1).min():.2f} ~ {Q_normal.norm(dim=-1).max():.2f}")
print(f"  K 模长范围: {K_large.norm(dim=-1).min():.2f} ~ {K_large.norm(dim=-1).max():.2f}")
print(f"  Attention score: {score_no_norm[0,0,0].tolist()}")
print(f"  Attention 分布:  {[f'{v:.3f}' for v in attn_no_norm[0,0,0].tolist()]}")
print(f"  → softmax 退化：大部分概率集中在个别位置")

print()
print("加 QK-Norm:")
print(f"  Q 模长范围: {Q_normed.norm(dim=-1).min():.2f} ~ {Q_normed.norm(dim=-1).max():.2f}")
print(f"  K 模长范围: {K_normed.norm(dim=-1).min():.2f} ~ {K_normed.norm(dim=-1).max():.2f}")
print(f"  Attention score: {score_with_norm[0,0,0].tolist()}")
print(f"  Attention 分布:  {[f'{v:.3f}' for v in attn_with_norm[0,0,0].tolist()]}")
print(f"  → 分布均匀，Attention 正常工作")

print()
print("关键观察：QK-Norm 把 Q、K 模长都拉到 ≈ 1，")
print("score 值域被限制在 [-1, 1] 附近，softmax 不再退化。")

In [ ]:
# === 带 QK-Norm 的 GQA 完整实现 ===

class GroupedQueryAttention_QKNorm(nn.Module):
    """
    GQA + QK-Norm（LLaMA 3 / Qwen 3 同款）
    
    相比标准 GQA，只多了两行 RMSNorm：一行给 Q，一行给 K。
    """
    def __init__(self, d_model, num_q_heads, num_kv_heads, eps=1e-6):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0
        
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.d_head = d_model // num_q_heads
        self.q_per_group = num_q_heads // num_kv_heads
        
        self.W_q = nn.Linear(d_model, num_q_heads * self.d_head, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(num_q_heads * self.d_head, d_model, bias=False)
        
        # QK-Norm：对 Q 和 K 各自做 RMSNorm
        self.q_norm = RMSNorm(self.d_head, eps=eps)
        self.k_norm = RMSNorm(self.d_head, eps=eps)
    
    def forward(self, x, mask=None):
        B, S, D = x.shape
        
        q = self.W_q(x).view(B, S, self.num_q_heads, self.d_head)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_head)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_head)
        
        # QK-Norm：在算 attention 之前，各自做 RMSNorm
        q = self.q_norm(q)
        k = self.k_norm(k)
        
        # GQA expand
        k = k[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)
        v = v[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)
        
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores + mask
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)


# 测试
d_model, num_q, num_kv = 16, 4, 2
attn_qknorm = GroupedQueryAttention_QKNorm(d_model, num_q, num_kv)
x = torch.randn(1, 6, d_model)
mask = torch.triu(torch.ones(6, 6) * float('-inf'), diagonal=1)
out = attn_qknorm(x, mask)

print("=== GQA + QK-Norm 测试 ===")
print(f"输入: {x.shape}, 输出: {out.shape}")
print(f"Q heads: {num_q}, KV heads: {num_kv}")
print(f"改动：比标准 GQA 多了 self.q_norm 和 self.k_norm 两行")
print("效果：LLaMA 3 和 Qwen 3 证明了这两行让深层训练更稳定")

### 8. 改进六：MHA → GQA → MLA — KV Cache 的终极压缩

**GQA 压缩了 KV 头数，还能再压吗？**

GQA 把 32 个 KV 头压缩到 4 个，KV Cache 变成 1/8。但每个 KV 头内部仍然是一个 `d_head` 维的完整向量——信息在「头数」维度上压缩了，但在「每头维度」上没有。

DeepSeek-V2 提出的 MLA（Multi-head Latent Attention）换了一个思路：不去减少头数，而是把整条 KV 信息流做**低秩压缩**。

```text
MHA/GQA 的做法：
  x → W_K → K (完整维度，存进 KV Cache)
  x → W_V → V (完整维度，存进 KV Cache)

MLA 的做法：
  x → W_down → c_KV (极小的 latent 向量，只存这个)
  推理时：c_KV → W_up_K → K
          c_KV → W_up_V → V
```

关键是 `c_KV` 的维度可以非常小——比如 d_model=512，d_head=128 时，c_KV 只需 64 维。这意味着 KV Cache 不再按 `num_heads × d_head` 存，而是按 `d_latent`（压缩维度）存。

**手算对比：MHA vs GQA vs MLA 的 KV Cache**

以一个小型模型为例：d_model=512, num_heads=8, d_head=64, seq_len=4096。


In [ ]:
# === MHA vs GQA vs MLA：KV Cache 终极对比 ===
print("=== KV Cache 三阶段对比：MHA → GQA → MLA ===")
print()

# 小型模型参数
d_model = 512
num_q_heads = 8
d_head = d_model // num_q_heads  # = 64
d_latent = 64  # MLA 压缩维度（远小于 d_model=512）
seq_len = 4096
dtype_bytes = 2  # bf16

print(f"模型: d_model={d_model}, Q heads={num_q_heads}, d_head={d_head}")
print(f"序列长度: {seq_len}, dtype: bf16")
print(f"MLA latent 维度: {d_latent}")
print()

# MHA: KV Cache = 2 × num_heads × d_head × seq_len × bytes
mha_kv = 2 * num_q_heads * d_head * seq_len * dtype_bytes

# GQA-4: 4 个 KV 头
gqa_kv_heads = 4
gqa_kv = 2 * gqa_kv_heads * d_head * seq_len * dtype_bytes

# GQA-2: 2 个 KV 头
gqa2_kv = 2 * 2 * d_head * seq_len * dtype_bytes

# MLA: 只存压缩后的 latent
mla_kv = 2 * d_latent * seq_len * dtype_bytes  # 存 c_KV（含 K 和 V 信息）

configs = [
    ("MHA", mha_kv),
    ("GQA-4", gqa_kv),
    ("GQA-2", gqa2_kv),
    ("MLA", mla_kv),
]

print(f"{'方案':<10s} {'KV Cache 大小':>16s} {'相对 MHA':>10s} {'说明'}")
print("-" * 65)

for name, size in configs:
    mb = size / (1024**2)
    ratio = size / mha_kv * 100
    note = ""
    if name == "MHA":
        note = "每个 Q 头独立 KV"
    elif name == "GQA-4":
        note = "4 组 Q 共享 KV"
    elif name == "GQA-2":
        note = "2 组 Q 共享 KV，效果开始掉"
    elif name == "MLA":
        note = f"低秩压缩到 {d_latent} 维"
    print(f"{name:<10s} {mb:>13.1f} MB {'(' + str(int(ratio)) + '%)':>11s} {note}")

print()
print("关键数字：")
print(f"  MHA → GQA-4: 省 {(1 - gqa_kv/mha_kv)*100:.0f}%")
print(f"  MHA → MLA:   省 {(1 - mla_kv/mha_kv)*100:.0f}%")
print(f"  GQA-4 → MLA: 省 {(1 - mla_kv/gqa_kv)*100:.0f}%")
print()
print("MLA 的魔力：d_latent << d_model（64 << 512），但信息损失很小")
print("因为 KV 矩阵本身就是低秩的——大部分信息可以用少量方向表示")

**MLA 的实现：核心就是一对上下投影**

MLA 和标准 Attention 的区别只在 KV 的来源：

```text
标准: K = W_K @ x,  V = W_V @ x  （直接从 x 投影）
MLA:  c_KV = W_down @ x          （先压缩到一个很小的 latent）
      K = W_up_K @ c_KV          （推理时从 latent 解压出 K）
      V = W_up_V @ c_KV          （推理时从 latent 解压出 V）
```

下面用极简实现展示这个机制。完整的 MLA 还涉及 RoPE 的解耦处理，这里先聚焦核心的低秩压缩思路。


In [ ]:
# === MLA 极简实现（展示低秩压缩的核心机制） ===

class MultiHeadLatentAttention(nn.Module):
    """
    MLA（Multi-head Latent Attention）极简版
    
    KV 先压缩到 d_latent 维的 latent 空间，
    推理时从 latent 解压出 K 和 V。
    
    参数:
        d_model: 模型维度
        num_heads: 注意力头数
        d_latent: KV 压缩维度（远小于 d_model）
    """
    def __init__(self, d_model, num_heads, d_latent):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_latent = d_latent
        
        # Q 照常投影
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        # KV 先压缩到 latent
        self.W_kv_down = nn.Linear(d_model, d_latent, bias=False)
        # 再从 latent 解压出 K 和 V
        self.W_k_up = nn.Linear(d_latent, d_model, bias=False)
        self.W_v_up = nn.Linear(d_latent, d_model, bias=False)
        # 输出投影
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        B, S, D = x.shape
        
        # Q：正常投影
        q = self.W_q(x).view(B, S, self.num_heads, self.d_head)
        
        # KV：先压缩到 latent，再解压
        c_kv = self.W_kv_down(x)           # [B, S, d_latent]
        k = self.W_k_up(c_kv).view(B, S, self.num_heads, self.d_head)
        v = self.W_v_up(c_kv).view(B, S, self.num_heads, self.d_head)
        
        # 标准的 scaled dot-product attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores + mask
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_o(out), c_kv  # 返回 c_kv 就是 KV Cache


# 测试 MLA
d_model = 64
num_heads = 4
d_latent = 16  # 压缩维度：16 << 64

mla = MultiHeadLatentAttention(d_model, num_heads, d_latent)
x = torch.randn(1, 6, d_model)
mask = torch.triu(torch.ones(6, 6) * float('-inf'), diagonal=1)

out, c_kv = mla(x, mask)

print("=== MLA 极简实现测试 ===")
print(f"d_model={d_model}, num_heads={num_heads}, d_latent={d_latent}")
print(f"输入: {x.shape}")
print(f"压缩 latent c_kv: {c_kv.shape}  ← 这就是 KV Cache 的大小")
print(f"输出: {out.shape}")
print()

# KV Cache 对比
mha_kv_per_token = 2 * num_heads * (d_model // num_heads)  # K+V 完整维度
mla_kv_per_token = d_latent  # 只存 latent

print(f"每个 token 的 KV Cache(元素数):")
print(f"  MHA: {mha_kv_per_token} (K+V, {num_heads} 头 × {d_model//num_heads} 维)")
print(f"  MLA: {mla_kv_per_token} (只有 c_kv, {d_latent} 维)")
print(f"  压缩比: {mla_kv_per_token/mha_kv_per_token:.1%}")
print()
print("MLA 的 trade-off：")
print(f"  训练时: 多了 W_kv_down, W_k_up, W_v_up 三个线性层（额外的参数和计算）")
print(f"  推理时: KV Cache 从 {mha_kv_per_token} 降到 {mla_kv_per_token} 个元素")
print(f"  → 同样的显存可以处理 {(mha_kv_per_token/mla_kv_per_token):.0f} 倍长的序列")

**附录：多路残差连接（mHC）— DeepSeek V4 的超深网络梯度控制**

前面讲了残差连接的演进：Post-Norm（原始 Transformer）→ Pre-Norm（LLaMA）→ DeepNorm（极深网络）。它们的共同点是一条残差路径配合一个归一化层，区别只在于归一化放在残差前还是后，以及归一化的力度。

DeepSeek V4 的 mHC（manifold-constrained hyper-connections）换了一个思路：不用一条残差路径，而是多条路径并行，每条路径有可学习的混合权重。

**多路残差的动机**

标准残差连接是：

$$x_{l+1} = x_l + f_l(x_l)$$

信息从第 l 层到第 l+1 层只有两种走法：原样通过（$x_l$，称为 identity 分支）或被变换（$f_l(x_l)$，称为 residual 分支）。而且两个分支的比例是固定的 1:1。在 100 层的网络中，每一层都用同样的 1:1 比例——网络没有能力自己决定「这一层应该多保留一些原信息，还是多做一些变换」。

mHC 给网络这个自由度：

$$x_{l+1} = \alpha_l \cdot x_l + \beta_l \cdot f_l(x_l)$$

其中 $\alpha_l, \beta_l$ 不是固定的，而是网络自己学习的参数。更一般地，mHC 维护 N 条并行的信息流（N 通常取 4~8），每条流有独立的残差混合系数：

```text
标准残差（1 条路径，固定 1:1）:
  x ──→ [Layer] ──→ × 1 ──┐
  │                        ├──→ + ──→ x'
  └────────────────────────┘
       identity 分支（固定 × 1）

mHC（N=4 条路径，可学习权重）:
  path_0: x ──→ [Layer] ──→ x α_0 ──┐
  path_1: x ──→ [Layer] ──→ x α_1 ──┤
  path_2: x ──→ [Layer] ──→ x α_2 ──┼──→ + ──→ x'
  path_3: x ──→ [Layer] ──→ x α_3 ──┘
  每个 α_i 是可学习参数，初始值保证整体增益为 1
```

每条路径经过相同的层变换，但乘以不同的混合系数。这些系数构成一个矩阵 H（hyper-connection matrix），在训练中学习和调整。

**流形约束**

如果让 α_0~α_3 完全自由学习，网络可能把它们全部推到极端值——全部变成 0（信息断流）或全部变得很大（梯度爆炸）。流形约束就是对 H 矩阵施加的限制：要求 H 的行或列满足某种正交性条件。

从数学上看，这个约束让 H 矩阵始终停留在一个「好」的流形上（比如 Stiefel 流形，即半正交矩阵的集合）。这就是名称中 "manifold-constrained" 的来源。实际操作中通常用正则化项或参数重投影来保证这个约束。

**在超深网络中的作用**

DeepSeek V4-Pro 有 1.6T 参数、数百层。在这个深度下，标准 Pre-Norm 虽然能让梯度传到深层，但「传多少比例」这件事完全由网络结构决定，而非由数据决定。mHC 让每一层自己学习最优的混合比例——浅层可能偏好大比例变换（提取特征），深层可能偏好大比例保留（微调已有的表示）。

从残差连接的演进历程来看：

```text
Post-Norm (2017)   残差后归一化，深层梯度消失
    ↓
Pre-Norm (2020)    残差前归一化，训练稳定但表示灵活性受限
    ↓
DeepNorm (2022)    更强的归一化力度，专为 100+ 层设计
    ↓
mHC (2025)         多路可学习残差，每一层有独立的混合策略
```

每一次演进都在解决同一个问题：信息如何在深层网络中既不消失也不失控地流动。mHC 是目前这一方向上前沿的工业方案。